In [67]:
# Imported all required dependencies
import torch 
import torch.nn as nn
import math

In [68]:
# js checking if the GPU is available for training.
device = torch.device('cuda'if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [69]:
#Making a class instance for selfattention machenism
class SelfAttention(nn.Module):
    def __init__(self, embed_size, heads):
        super(SelfAttention,self).__init__()
        self.embed_size = embed_size # dimension of each token embedding (NOT number of tokens)
        self.heads = heads # No of parallel attentions
        self.head_dim = embed_size//heads # dimension handled by each attention head

        assert (self.head_dim*heads==embed_size) # Just ensuring the split is correct if not the code will break
        self.value = nn.Linear(embed_size,embed_size,bias=False) 
        self.key = nn.Linear(embed_size,embed_size,bias=False)
        self.query = nn.Linear(embed_size,embed_size,bias=False)
        self.fc_out = nn.Linear(embed_size,embed_size,bias=False)# concatinate head otputs :-> like think of all 8 heads here are expert and gave some output this fc_out just concatinate all the outputs
    
#Initially, all tokens share the same embedding representation X, but after passing through different linear layers (Wq, Wk, Wv), they are projected into different semantic spaces, allowing the model to learn distinct roles for query, key, and value.

# bias shift the vector by some constant like Output = XWq + c ...Bias is removed because attention relies on dot-product similarity between Query and Key, and adding a bias term introduces unnecessary shifts that don’t improve this similarity while increasing parameters.

#🧠 Deep intuition
#Same word:-> "bank" Different meanings:-> river bank ,financial bank
#👉 Linear layers help transform embeddings into different semantic spaces
    def forward(self,value,key,query,mask=None):
        #Query = [N,query_len,embed_size] N->No of trainable examples in a batch...  , Query_len is no of token in query and embed_size is dim of embedding of token
        N = query.shape[0]
        query_len,key_len,value_len = query.shape[1],key.shape[1],value.shape[1]
        value = self.value(value) # Value = [N,value_len,embed_size]
        query = self.query(query) #query = [N,query_len,embed_size]
        key = self.key(key) #key = [N,key_len,embed_size]
    
    # Split the embed size in self head pieves for query,key and value 
        value= value.reshape(N,value_len,self.heads,self.head_dim)
        key = key.reshape(N,key_len,self.heads,self.head_dim)
        query = query.reshape(N,query_len,self.heads,self.head_dim)
    # Key shape = [N, key_len , heads ,head_dim] -> nkhd
    # query shape = [N, query_len , heads ,head_dim] -> nqhd
    # Attention_score shape = [N,heads , query_len , key_len,] -> nhqk

        attn_scores = torch.einsum("nqhd,nkhd->nhqk", query, key) # dim 1 ,dim2 , dim3

    # Masking is done because all the sentenses are not of same size and we sometime trancate some vector embedding or just padd some vector embedding to make them of same lenght..Genrally we do both trancation and padding and change the missing space by a large negative number so that it become zero when passed through softmax
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask==0 ,float("-le20"))

    # Now we have all are input like QKV passed them throgh linear layer and then reshape them in heads and head_dim then matrix multiplication for attention scores and now we will pass them throgh softmax later
        attention = torch.softmax(attn_scores/self.head_dim**(1/2),dim = 3) 

    # attention shape: (N, heads, query_len, key_len)
    # values shape: (N, value_len, heads, heads_dim)
    # out after matrix multiply: (N, query_len, heads, head_dim), then
    # we reshape and concatenate the last two dimensions.
        out = torch.einsum('nhql,nlhd->nqhd',[attention,value]).reshape([N,query_len,self.head_dim*self.heads])

        out = self.fc_out(out)
        return out
        

In [70]:
# Now we have muldi head attention mechanism lets move to the add norm and the feed forward layer okayyy

# This is a TranformerBlock Like it includes the add norm and the feed forwardlayer part and this same part is with the decoder block so it can be used further also
class TransformerBlock(nn.Module):
    def __init__(self, embed_size,heads,forward_expansion,dropout):
        super(TransformerBlock,self).__init__() 
        self.attention = SelfAttention(embed_size,heads)
        self.norm1 = nn.LayerNorm(embed_size) # This is normalization layer 1 for the multi head attention and norm 2 will be used after the feed forward layer
        self.norm2 = nn.LayerNorm(embed_size) # This is norm2 will be used after the feed forward layer

        # What feed forward layer does see from the very first part of our court order very first part of the transformer Over all the output like query key and value are linear and when it comes to a linear function it can't capture the more complex task what feed forward layer does is it add some nonlinearity to the output part of the multi head attention so it can learn more complex AH part or recognize more complex things it first projects input to high dimension like D dimension to 40 dimension then it again converges to the initial dimension like linear to nonlinear so it can learn more complex tasks then again linear that's what it does and what normalization do is see we have a input and it will be in a fixed like format most of the time multi ad attention or the feed forward layer manipulate it or fluctuate it OK By that it deviates from its normal representation to various types normalization add norm again convert it to the initial representation part so there is no mess up in the input or the output of the mechanism 
        self.feed_forward = nn.Sequential(
            nn.Linear(embed_size,forward_expansion*embed_size),
            nn.ReLU(),
            nn.Linear(forward_expansion*embed_size,embed_size)
        )
        # Dropout is a regularization function what it does is it drop out some neurons to prevent the overfitting By dropping out some neurons it adds a bit more complexity to the neural network How because when it comes to dropping out some neurons our mechanism or model will not rely only just some neurons it will move one to another to another to another that's how it can learn more complex task and the complexity is bit more added to it.
        self.dropout = nn.Dropout(dropout)

    def forward(self,value,key,query,mask):
        attention = self.attention(value,key,query,mask)
        x = self.dropout(self.norm1(attention+query))
        forward = self.feed_forward(x)
        out = self.dropout(self.norm2(x+ forward))
        return out



In [71]:
class PositionalEncoding(nn.Module):
    """
    Positional Encoding Module

    Adds positional information to token embeddings so that the Transformer
    can understand the order of tokens in a sequence.

    This implementation uses fixed sinusoidal functions (no learnable parameters),
    as introduced in the "Attention Is All You Need" paper.

    Key Idea:
    - Even indices (0,2,4,...) use sin()
    - Odd indices (1,3,5,...) use cos()
    - Different dimensions use different frequencies

    Shape Details:
    - Input:  (batch_size, seq_len, embed_size)
    - Output: (batch_size, seq_len, embed_size)
    """

    def __init__(self, embed_size, max_len=5000):
        """
        Args:
            embed_size (int): Dimension of token embeddings (d_model)
            max_len (int): Maximum sequence length supported
        """
        super(PositionalEncoding, self).__init__()

        # Create a matrix of shape (max_len, embed_size)
        pe = torch.zeros(max_len, embed_size)

        # Create position indices (0, 1, 2, ..., max_len-1)
        position = torch.arange(0, max_len).unsqueeze(1)
        # Shape: (max_len, 1)

        # Compute the denominator term for scaling frequencies
        # This creates different frequencies for each dimension
        div_term = torch.exp(
            torch.arange(0, embed_size, 2) * -(math.log(10000.0) / embed_size)
        )
        # Shape: (embed_size/2)

        # Apply sine to even indices (0, 2, 4, ...)
        pe[:, 0::2] = torch.sin(position * div_term)

        # Apply cosine to odd indices (1, 3, 5, ...)
        pe[:, 1::2] = torch.cos(position * div_term)

        # Add batch dimension → (1, max_len, embed_size)
        pe = pe.unsqueeze(0)

        # Register as buffer (not a parameter, but moves with model)
        self.register_buffer("pe", pe)

    def forward(self, x):
        """
        Forward pass

        Args:
            x (Tensor): Input embeddings
                        Shape: (batch_size, seq_len, embed_size)

        Returns:
            Tensor: Embeddings with positional encoding added
                    Shape: (batch_size, seq_len, embed_size)
        """

        # Add positional encoding to input embeddings
        # Only take required sequence length
        x = x + self.pe[:, :x.shape[1]]

        return x
    